In [1]:
# module for building the pyomo model
from pyomo.environ import *
import pyomo.environ as pe

# module for solving the pyomo model
import pyomo.opt as po

import json

In [2]:
model = pe.ConcreteModel()

# Cargar Datos

In [3]:
with open("../data/processed_necesidades.json", "r", encoding="utf-8") as f:
    necesidades_dict = json.load(f)
with open("../data/processed_planes.json", "r", encoding="utf-8") as f:
    planes_dict = json.load(f)
with open("../data/processed_relacion_necesidad_plan.json", "r", encoding="utf-8") as f:
    relaciones_dict = json.load(f)

In [5]:
# Situación de prueba
N_set = ['1.2.2.', '2.2.1.', '4.1.1.']
R_set = [(need, plan) for need, plans in relaciones_dict.items() if need in N_set for plan in plans]
A_set = list({plan for (_, plan) in R_set})

In [6]:
necesidades = {n: necesidades_dict[n]['necesidad'] for n in N_set}
acciones = {a: planes_dict[a]['descripcion'] for a in A_set}

# Conjuntos

In [7]:
# Necesidades (N)
model.n = pe.Set(initialize=N_set)
# Acciones/Planes (A)
model.a = pe.Set(initialize=A_set)
# Relaciones válidas (R)
model.r = pe.Set(within=model.n * model.a, initialize=R_set)

# Parámetros

In [8]:
u = {n: necesidades_dict[n]['urgencia'] for n in N_set}
imp = {n: necesidades_dict[n]['importancia'] for n in N_set}
p = {a: planes_dict[a]['plazo_ejecucion'] for a in A_set}
c = {a: planes_dict[a]['complejidad'] for a in A_set}
# Condiciones de aplicabilidad (1 = aplicable, 0 = no aplicable)
# cond = {'A1':1, 'A2':1, 'A3':1, 'A4':1,
#         'A5':1, 'A6':1, 'A7':1, 'A8':1, 'A9':1, 'A10':1}

model.u = pe.Param(model.n, initialize=u)
model.imp = pe.Param(model.n, initialize=imp)
model.p = pe.Param(model.a, initialize=p)
model.c = pe.Param(model.a, initialize=c)
# model.cond = pe.Param(model.a, initialize=cond)

# Párametros de Ponderación

In [9]:
alpha = 4   # Peso urgencia
beta  = 3   # Peso importancia
gamma = 0.5 # Penalización plazo
delta = 1   # Penalización complejidad

# Variables

In [10]:
# x_ij: acción j asignada a necesidad i
model.x = pe.Var(model.r, within=pe.Binary)
# y_j: la acción j es seleccionada
model.y = pe.Var(model.a, within=pe.Binary)

# Función de Peso

In [11]:
def weight_rule(model,i,j):
    return alpha*model.u[i] + beta*model.imp[i] - gamma*model.p[j] - delta*model.c[j]
model.w = pe.Param(model.n, model.a,
                   initialize={(i,j):weight_rule(model,i,j) for i in model.n for j in model.a})


# Función Objetivo

In [12]:
def obj_rule(model):
    return sum(model.w[i,j]*model.x[(i,j)] for (i,j) in model.r)
model.obj = pe.Objective(rule=obj_rule, sense=pe.maximize)

# Restricciones

In [13]:
# Cobertura de necesidades
def cobertura_rule(model,i):
    return sum(model.x[(i,j)] for j in model.a if (i,j) in model.r) >= 1
model.cobertura = pe.Constraint(model.n, rule=cobertura_rule)

In [14]:
# Coherencia de selección
def coherencia_rule(model, i, j):
    if (i,j) in model.r:
        return model.x[(i,j)] <= model.y[j]
    return pe.Constraint.Skip
model.coherencia = pe.Constraint(model.n, model.a, rule=coherencia_rule)


In [15]:
# Aplicabilidad
# def aplicabilidad_rule(model, j):
#     return model.y[j] <= model.cond[j]
# model.aplicabilidad = pe.Constraint(model.a, rule=aplicabilidad_rule)


In [29]:
# Minimalidad de acciones seleccionadas
M = 4 # máximo de acciones (ejemplo)
model.minimalidad = pe.Constraint(expr=sum(model.y[j] for j in model.a) <= M)


'pyomo.core.base.constraint.ScalarConstraint'>) on block unknown with a new
Component (type=<class
'pyomo.core.base.constraint.AbstractScalarConstraint'>). This is usually
indicative of a modelling error. To avoid this warning, use
block.del_component() and block.add_component().


# Resultado

In [30]:
solver = pe.SolverFactory('gurobi')
result = solver.solve(model, tee=True)

Read LP format model from file C:\Users\carlo\AppData\Local\Temp\tmpw_oltpp7.pyomo.lp
Reading time = 0.02 seconds
x1: 14 rows, 20 columns, 40 nonzeros
Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: Intel(R) Core(TM) i7-8550U CPU @ 1.80GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Academic license 2557253 - for non-commercial use only - registered to 20___@alu.comillas.edu
Optimize a model with 14 rows, 20 columns and 40 nonzeros
Model fingerprint: 0xc2cba08a
Variable types: 0 continuous, 20 integer (20 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [2e+00, 1e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 4e+00]
Found heuristic solution: objective 21.0000000
Presolve removed 11 rows and 13 columns
Presolve time: 0.00s
Presolved: 3 rows, 7 columns, 14 nonzeros
Variable types: 0 continuous, 7 integer (5 binary)
Found h

In [31]:
print('Acciones recomendadas:')
for j in model.a:
    if value(model.y[j]) > 0.5:
        print(acciones[j])
print('\nAsignación necesidad – acción:')
for (i,j) in model.r:
    if value(model.x[(i,j)]) > 0.5:
        print(f'(Necesidad) {necesidades[i]} -> (Acción) {acciones[j]}')
print('\nValor objetivo:',round(value(model.obj),2))

Acciones recomendadas:
Intenta reducir los stocks, acortar tus plazos de cobro y alargar los plazos de pago, renegociando con tus clientes y proveeodres las condiciones y medios de pago.
Gestionar ponderar la proporción de concentración por debajo del 20% por cliente, incoporando nuevos clientes. Una concentracion en clientes es un factor de riesgo relevante, pues condicona precios a la baja y exigencias en plazos de pago
Aprovechar los clientes mas consolidados y vinculados/condicionados por relacion para incrementar margen 
Identificar los proveedores mas consolidados y con mejor relacion para reducir precios y alargar plazos de pago. Gestionar acuerdos marco de compra a medio y largo plazo que permitan asegurar o reducir precio con esuemas de rapell por volumen. 

Asignación necesidad – acción:
(Necesidad) Los financiadores pueden pensar que no has aportando suficiente para el tamaño actual de tu empresa -> (Acción) Intenta reducir los stocks, acortar tus plazos de cobro y alargar l